# 🥗 Fit Smart — EDA Notebook
**Data Scientist Pipeline — Step 2: Exploratory Data Analysis**

Dataset yang dianalisis:
- Indonesian Food & Drink Nutrition Dataset (Kaggle)
- Indonesian Food Image Dataset (Mendeley)

---

## ⚙️ 0. KONFIGURASI PATH — EDIT DI SINI DULU!
Ganti path di bawah sesuai lokasi folder kamu di Google Drive.

In [ ]:
# ============================================================
# KONFIGURASI PATH LOKAL KOMPUTER
# ============================================================

# Path ke file CSV / Excel nutrition dataset
NUTRITION_CSV = r'd:\CodingCamp\CAPSTONE\nutrition.csv'

# Path ke folder gambar
IMAGES_DIR    = r'd:\CodingCamp\CAPSTONE\Indonesian Food Image'

# ============================================================
print('✅ Path dikonfigurasi')
print(f'   Nutrition : {NUTRITION_CSV}')
print(f'   Images    : {IMAGES_DIR}')

## 📦 1. Mount Google Drive & Install Library

In [ ]:
# Karena dijalankan di komputer lokal, bagian Google Colab ini dinonaktifkan
# from google.colab import drive
# drive.mount('/content/drive')
print('✅ Mode Lokal: Tidak perlu mount Google Drive')

In [ ]:
!pip install -q matplotlib seaborn pandas pillow

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi']  = 120
plt.rcParams['font.size']   = 11
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Semua library berhasil di-import')

---
## 🧾 2. Load & Preview — Indonesian Nutrition Dataset

In [ ]:
# Load dataset — coba CSV dulu, fallback ke Excel
try:
    df = pd.read_csv(NUTRITION_CSV)
except Exception:
    df = pd.read_excel(NUTRITION_CSV)

print(f'Shape   : {df.shape[0]:,} baris × {df.shape[1]} kolom')
print(f'Kolom   : {list(df.columns)}')
df.head(10)

In [ ]:
# Ringkasan statistik
df.describe(include='all').round(2)

---
## 🔍 3. Cek Kualitas Data — Missing Values & Duplikat

In [ ]:
# ── Missing values ──────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah Missing': missing,
                            'Persentase (%)': missing_pct})
missing_df = missing_df[missing_df['Jumlah Missing'] > 0].sort_values('Persentase (%)', ascending=False)

print('=== Missing Values ===')
if missing_df.empty:
    print('✅ Tidak ada missing values!')
else:
    display(missing_df)

# ── Duplikat ────────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f'\n=== Duplikat ===')
print(f'Baris duplikat : {n_dup} ({n_dup/len(df)*100:.2f}%)')

# ── Visualisasi missing values ───────────────────────────────
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing_df)*0.5)))
    missing_df['Persentase (%)'].plot(kind='barh', ax=ax, color='#E24B4A')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Kolom dengan Missing Values')
    plt.tight_layout()
    plt.show()

---
## 🔧 4. Deteksi & Standarisasi Kolom Kalori
> Notebook ini auto-detect nama kolom kalori — tidak perlu edit manual.

In [ ]:
# Auto-detect kolom kalori & nutrisi lainnya
CALORIE_KEYWORDS = ['calori', 'kalori', 'kcal', 'energy', 'energi', 'cal']
PROTEIN_KEYWORDS = ['protein', 'protei']
FAT_KEYWORDS     = ['fat', 'lemak', 'lipid']
CARB_KEYWORDS    = ['carb', 'karbo', 'karbohidrat', 'carbohydrate', 'sugar', 'gula']
NAME_KEYWORDS    = ['name', 'nama', 'food', 'makanan', 'item']

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in col.lower():
                return col
    return None

COL_NAME    = find_col(df, NAME_KEYWORDS)
COL_CAL     = find_col(df, CALORIE_KEYWORDS)
COL_PROTEIN = find_col(df, PROTEIN_KEYWORDS)
COL_FAT     = find_col(df, FAT_KEYWORDS)
COL_CARB    = find_col(df, CARB_KEYWORDS)

print('=== Kolom yang terdeteksi ===')
for label, col in [('Nama makanan', COL_NAME),
                   ('Kalori',       COL_CAL),
                   ('Protein',      COL_PROTEIN),
                   ('Lemak',        COL_FAT),
                   ('Karbohidrat',  COL_CARB)]:
    status = '✅' if col else '⚠️  tidak ditemukan'
    print(f'  {label:15s} → {col or status}')

if not COL_CAL:
    print('\n⚠️  Kolom kalori tidak terdeteksi otomatis.')
    print('Silakan isi manual: COL_CAL = "nama_kolom_kalori"')

---
## 📊 5. Distribusi Kalori

In [ ]:
if COL_CAL:
    cal = df[COL_CAL].dropna()
    cal_num = pd.to_numeric(cal, errors='coerce').dropna()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Histogram
    axes[0].hist(cal_num, bins=40, color='#1D9E75', edgecolor='white', linewidth=0.4)
    axes[0].set_title('Distribusi Kalori')
    axes[0].set_xlabel('Kalori (kcal)')
    axes[0].set_ylabel('Jumlah item')
    axes[0].axvline(cal_num.mean(),   color='#E24B4A', linestyle='--', label=f'Mean={cal_num.mean():.0f}')
    axes[0].axvline(cal_num.median(), color='#185FA5', linestyle='--', label=f'Median={cal_num.median():.0f}')
    axes[0].legend(fontsize=9)

    # Boxplot
    axes[1].boxplot(cal_num, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#E1F5EE'),
                    medianprops=dict(color='#0F6E56', linewidth=2))
    axes[1].set_title('Boxplot Kalori')
    axes[1].set_ylabel('Kalori (kcal)')
    axes[1].set_xticks([])

    # Range bucket
    buckets = pd.cut(cal_num,
                     bins=[0, 100, 200, 300, 400, 500, 700, 1000, float('inf')],
                     labels=['0–100','100–200','200–300','300–400',
                             '400–500','500–700','700–1000','1000+'])
    bucket_counts = buckets.value_counts().sort_index()
    bucket_counts.plot(kind='bar', ax=axes[2], color='#378ADD', edgecolor='white')
    axes[2].set_title('Distribusi per Range Kalori')
    axes[2].set_xlabel('Range (kcal)')
    axes[2].set_ylabel('Jumlah item')
    axes[2].tick_params(axis='x', rotation=35)

    plt.suptitle('Analisis Distribusi Kalori — Indonesian Nutrition Dataset', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    print(f'\n📊 Statistik kalori:')
    print(f'  Min    : {cal_num.min():.1f} kcal')
    print(f'  Max    : {cal_num.max():.1f} kcal')
    print(f'  Mean   : {cal_num.mean():.1f} kcal')
    print(f'  Median : {cal_num.median():.1f} kcal')
    print(f'  Std    : {cal_num.std():.1f} kcal')

    # Deteksi outlier (IQR method)
    Q1, Q3 = cal_num.quantile(0.25), cal_num.quantile(0.75)
    IQR = Q3 - Q1
    outliers = cal_num[(cal_num < Q1 - 1.5*IQR) | (cal_num > Q3 + 1.5*IQR)]
    print(f'\n⚠️  Outlier terdeteksi (IQR method): {len(outliers)} item')
    if len(outliers) > 0 and COL_NAME:
        display(df.loc[outliers.index, [COL_NAME, COL_CAL]].head(10))
else:
    print('⚠️  Kolom kalori belum terdeteksi, isi COL_CAL manual.')

---
## 🔗 6. Korelasi Antar Nutrisi

In [ ]:
nutri_cols = [c for c in [COL_CAL, COL_PROTEIN, COL_FAT, COL_CARB] if c]

if len(nutri_cols) >= 2:
    nutri_df = df[nutri_cols].apply(pd.to_numeric, errors='coerce').dropna()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Heatmap korelasi
    corr = nutri_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=axes[0], annot=True, fmt='.2f',
                cmap='RdYlGn', center=0, mask=mask,
                linewidths=0.5, square=True, vmin=-1, vmax=1)
    axes[0].set_title('Korelasi Antar Nutrisi')

    # Scatter: kalori vs nutrisi utama
    if COL_CAL and COL_FAT:
        axes[1].scatter(nutri_df[COL_FAT], nutri_df[COL_CAL],
                        alpha=0.4, color='#BA7517', s=25, edgecolors='none')
        axes[1].set_xlabel(f'Lemak ({COL_FAT})')
        axes[1].set_ylabel(f'Kalori ({COL_CAL})')
        axes[1].set_title('Kalori vs Lemak')
    elif COL_CAL and COL_CARB:
        axes[1].scatter(nutri_df[COL_CARB], nutri_df[COL_CAL],
                        alpha=0.4, color='#534AB7', s=25, edgecolors='none')
        axes[1].set_xlabel(f'Karbohidrat ({COL_CARB})')
        axes[1].set_ylabel(f'Kalori ({COL_CAL})')
        axes[1].set_title('Kalori vs Karbohidrat')

    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Perlu minimal 2 kolom nutrisi untuk analisis korelasi.')

---
## 🍽️ 7. Top & Bottom Makanan Berdasarkan Kalori

In [ ]:
if COL_CAL and COL_NAME:
    df_clean = df[[COL_NAME, COL_CAL]].copy()
    df_clean[COL_CAL] = pd.to_numeric(df_clean[COL_CAL], errors='coerce')
    df_clean = df_clean.dropna().drop_duplicates(subset=COL_NAME)
    df_clean = df_clean[df_clean[COL_CAL] > 0].sort_values(COL_CAL)

    N = 15
    top    = df_clean.tail(N).sort_values(COL_CAL, ascending=True)
    bottom = df_clean.head(N).sort_values(COL_CAL, ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].barh(bottom[COL_NAME], bottom[COL_CAL], color='#9FE1CB', edgecolor='white')
    axes[0].set_title(f'Top {N} Makanan RENDAH Kalori')
    axes[0].set_xlabel('Kalori (kcal)')

    axes[1].barh(top[COL_NAME], top[COL_CAL], color='#E24B4A', edgecolor='white')
    axes[1].set_title(f'Top {N} Makanan TINGGI Kalori')
    axes[1].set_xlabel('Kalori (kcal)')

    plt.tight_layout()
    plt.show()
else:
    print('⚠️  COL_NAME atau COL_CAL tidak ditemukan.')

---
## 🖼️ 8. Audit Dataset Gambar (Mendeley)

In [ ]:
img_dir = Path(IMAGES_DIR)

if not img_dir.exists():
    print(f'❌ Folder tidak ditemukan: {IMAGES_DIR}')
    print('Pastikan IMAGES_DIR sudah benar di cell konfigurasi.')
else:
    # Inventaris gambar per kelas
    class_counts = defaultdict(int)
    supported    = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

    # Cek apakah ada subfolder train/test/val
    has_splits = any((img_dir / split).exists() for split in ['train', 'test', 'val'])
    
    subdirs = []
    if has_splits:
        for split in ['train', 'test', 'val']:
            split_dir = img_dir / split
            if split_dir.exists():
                subdirs.extend([d for d in split_dir.iterdir() if d.is_dir()])
        print(f'✅ Struktur split dataset (train/test/val) terdeteksi')
    else:
        subdirs = [d for d in img_dir.iterdir() if d.is_dir()]

    if subdirs:
        for cls_dir in subdirs:
            imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in supported]
            class_counts[cls_dir.name] += len(imgs)
        print(f'✅ Ditemukan {len(set(class_counts.keys()))} kelas makanan unik')
    else:
        imgs = [f for f in img_dir.iterdir() if f.suffix.lower() in supported]
        class_counts['(flat)'] = len(imgs)
        print(f'⚠️  Folder flat — {len(imgs)} gambar ditemukan (tidak ada subfolder kelas)')

    total_imgs = sum(class_counts.values())
    print(f'Total gambar : {total_imgs:,}')
    print(f'Jumlah kelas : {len(class_counts)}')
    print(f'Rata-rata    : {total_imgs/max(len(class_counts),1):.0f} gambar/kelas')

    # Bar chart distribusi per kelas
    if len(class_counts) > 1:
        fig, ax = plt.subplots(figsize=(10, max(4, len(class_counts)*0.45)))
        counts_sorted = dict(sorted(class_counts.items(), key=lambda x: x[1]))
        colors_bar = ['#E24B4A' if v < total_imgs/len(class_counts)*0.5 else '#1D9E75'
                      for v in counts_sorted.values()]
        ax.barh(list(counts_sorted.keys()), list(counts_sorted.values()),
                color=colors_bar, edgecolor='white')
        ax.axvline(total_imgs/len(class_counts), color='#185FA5',
                   linestyle='--', label='Rata-rata')
        ax.set_xlabel('Jumlah gambar')
        ax.set_title('Distribusi Total Gambar per Kelas (Train+Test)')
        ax.legend()
        plt.tight_layout()
        plt.show()

        # Class imbalance ratio
        max_cls = max(class_counts.values())
        min_cls = min(class_counts.values())
        print(f'\n⚖️  Class imbalance ratio: {max_cls/max(min_cls,1):.1f}x')
        if max_cls/max(min_cls,1) > 3:
            print('   ⚠️  Imbalance cukup besar — perlu data augmentation untuk kelas minor!')

---
## 🔬 9. Audit Kualitas Gambar (Resolusi, Format, Corrupt)

In [ ]:
img_dir = Path(IMAGES_DIR)
supported = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

all_images = list(img_dir.rglob('*'))
all_images = [f for f in all_images if f.suffix.lower() in supported]

MAX_AUDIT = 500  # audit max 500 gambar untuk kecepatan
sample = all_images[:MAX_AUDIT]

widths, heights, formats, corrupt = [], [], [], []

for fpath in sample:
    try:
        with Image.open(fpath) as img:
            widths.append(img.width)
            heights.append(img.height)
            formats.append(img.format or fpath.suffix.upper())
    except Exception:
        corrupt.append(str(fpath))

print(f'Gambar diaudit  : {len(sample)} (dari total {len(all_images):,})')
print(f'Gambar corrupt  : {len(corrupt)} ({len(corrupt)/len(sample)*100:.1f}%)')

if widths:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Scatter resolusi
    axes[0].scatter(widths, heights, alpha=0.3, color='#534AB7', s=10, edgecolors='none')
    axes[0].axvline(224, color='#E24B4A', linestyle='--', label='224px (target)')
    axes[0].axhline(224, color='#E24B4A', linestyle='--')
    axes[0].set_xlabel('Lebar (px)')
    axes[0].set_ylabel('Tinggi (px)')
    axes[0].set_title('Distribusi Resolusi Gambar')
    axes[0].legend()

    # Histogram lebar
    axes[1].hist(widths, bins=30, color='#378ADD', edgecolor='white')
    axes[1].axvline(224, color='#E24B4A', linestyle='--', label='224px target')
    axes[1].set_title('Distribusi Lebar Gambar')
    axes[1].set_xlabel('Lebar (px)')
    axes[1].legend()

    # Format gambar
    fmt_counts = pd.Series(formats).value_counts()
    axes[2].pie(fmt_counts.values, labels=fmt_counts.index, autopct='%1.1f%%',
                colors=['#1D9E75','#378ADD','#BA7517','#993556','#5F5E5A'])
    axes[2].set_title('Format Gambar')

    plt.tight_layout()
    plt.show()

    pct_small = sum(1 for w,h in zip(widths,heights) if w<224 or h<224) / len(widths) * 100
    print(f'\n📐 Gambar < 224px (perlu upscale): {pct_small:.1f}%')
    print(f'   Min resolusi: {min(widths)}×{min(heights)} px')
    print(f'   Max resolusi: {max(widths)}×{max(heights)} px')
    print(f'   Rata-rata  : {int(np.mean(widths))}×{int(np.mean(heights))} px')

if corrupt:
    print(f'\n❌ File corrupt:')
    for c in corrupt[:10]:
        print(f'   {c}')

---
## 🖼️ 10. Visual Sample Gambar per Kelas

In [ ]:
img_dir = Path(IMAGES_DIR)
supported = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

# Kumpulkan folder kelas (dukung struktur train/test maupun langsung)
class_dirs_dict = {}
has_splits = any((img_dir / split).exists() for split in ['train', 'test', 'val'])

if has_splits:
    for split in ['train', 'test', 'val']:
        split_dir = img_dir / split
        if split_dir.exists():
            for d in split_dir.iterdir():
                if d.is_dir() and d.name not in class_dirs_dict:
                    class_dirs_dict[d.name] = d
else:
    for d in img_dir.iterdir():
        if d.is_dir():
            class_dirs_dict[d.name] = d

subdirs = sorted(list(class_dirs_dict.values()), key=lambda x: x.name)

if not subdirs:
    print('⚠️  Tidak ada subfolder kelas — gambar dalam satu folder flat.')
    print('Tampilkan 5 gambar pertama:')
    flat_imgs = [f for f in img_dir.iterdir() if f.suffix.lower() in supported][:5]
    if flat_imgs:
        fig, axes = plt.subplots(1, len(flat_imgs), figsize=(15, 3))
        if len(flat_imgs) == 1: axes = [axes]
        for ax, fp in zip(axes, flat_imgs):
            ax.imshow(Image.open(fp))
            ax.set_title(fp.name[:20], fontsize=8)
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print('Tidak ada gambar ditemukan.')
else:
    N_SHOW = min(len(subdirs), 10)  # max 10 kelas
    IMGS_PER_CLASS = 4

    fig, axes = plt.subplots(N_SHOW, IMGS_PER_CLASS,
                              figsize=(IMGS_PER_CLASS*3, N_SHOW*2.5))
    if N_SHOW == 1:
        axes = [axes]

    for row, cls_dir in enumerate(subdirs[:N_SHOW]):
        imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in supported]
        sample_imgs = imgs[:IMGS_PER_CLASS]

        for col in range(IMGS_PER_CLASS):
            ax = axes[row][col]
            if col < len(sample_imgs):
                try:
                    ax.imshow(Image.open(sample_imgs[col]))
                except Exception:
                    ax.text(0.5, 0.5, 'Error', ha='center', va='center')
            ax.axis('off')
            if col == 0:
                ax.set_title(f'{cls_dir.name}\n({len(imgs)} imgs)',
                             fontsize=9, loc='left')

    plt.suptitle('Sample Gambar per Kelas', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 🔗 11. Pengecekan Match Nama: Nutrition ↔ Gambar
> Ini langkah kritis — cek berapa banyak nama makanan di nutrition dataset yang bisa ditemukan di folder gambar.

In [ ]:
img_dir = Path(IMAGES_DIR)

# Ambil nama kelas (dukung struktur train/test)
class_names = set()
has_splits = any((img_dir / split).exists() for split in ['train', 'test', 'val'])

if has_splits:
    for split in ['train', 'test', 'val']:
        split_dir = img_dir / split
        if split_dir.exists():
            for d in split_dir.iterdir():
                if d.is_dir():
                    class_names.add(d.name.lower().replace(' ','_'))
else:
    for d in img_dir.iterdir():
        if d.is_dir():
            class_names.add(d.name.lower().replace(' ','_'))

subdirs = list(class_names)

if COL_NAME and subdirs:
    food_names = df[COL_NAME].dropna().str.lower().str.replace(' ','_').str.strip().unique()

    matched   = [n for n in food_names if any(n in s or s in n for s in subdirs)]
    unmatched = [n for n in food_names if n not in matched]

    print(f'Nama di nutrition CSV : {len(food_names)}')
    print(f'Folder kelas gambar   : {len(subdirs)}')
    print(f'✅ Match               : {len(matched)} ({len(matched)/len(food_names)*100:.1f}%)')
    print(f'⚠️  Tidak match        : {len(unmatched)} ({len(unmatched)/len(food_names)*100:.1f}%)')

    if unmatched:
        print(f'\n10 nama pertama yang tidak match di folder gambar:')
        for n in unmatched[:10]:
            print(f'  - {n}')

    # Visualisasi
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.pie([len(matched), len(unmatched)],
           labels=['Match', 'Tidak match'],
           autopct='%1.1f%%',
           colors=['#1D9E75', '#E24B4A'],
           startangle=90)
    ax.set_title('Match Nama Makanan\nNutrition CSV ↔ Folder Gambar')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Perlu COL_NAME terdeteksi dan folder gambar berupa subfolder kelas.')

---
## 📋 12. Ringkasan EDA & Rekomendasi Preprocessing

In [ ]:
print('=' * 55)
print('  RINGKASAN EDA — FIT SMART FOOD DATASET')
print('=' * 55)

print('\n[NUTRITION DATASET]')
print(f'  Total item      : {len(df):,}')
if COL_CAL:
    cal_num = pd.to_numeric(df[COL_CAL], errors='coerce')
    print(f'  Kalori mean     : {cal_num.mean():.1f} kcal')
    print(f'  Missing kalori  : {cal_num.isnull().sum()} item')
    zero_cal = (cal_num == 0).sum()
    print(f'  Kalori = 0      : {zero_cal} item  {"⚠️  PERLU DIHAPUS" if zero_cal > 0 else "✅"}')
n_dup = df.duplicated().sum()
print(f'  Duplikat        : {n_dup}  {"⚠️  PERLU DIHAPUS" if n_dup > 0 else "✅"}')

print('\n[IMAGE DATASET]')
img_dir = Path(IMAGES_DIR)
if img_dir.exists():
    all_imgs = list(img_dir.rglob('*'))
    all_imgs = [f for f in all_imgs if f.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    n_cls = len([d for d in img_dir.iterdir() if d.is_dir()])
    print(f'  Total gambar    : {len(all_imgs):,}')
    print(f'  Jumlah kelas    : {n_cls}')

print('\n[REKOMENDASI LANGKAH PREPROCESSING]')
steps = [
    '1. Drop baris kalori = 0 atau NaN dari nutrition CSV',
    '2. Lowercase & strip nama makanan (hapus spasi lebih)',
    '3. Hapus duplikat berdasarkan nama makanan',
    '4. Fuzzy match nama makanan CSV → folder gambar',
    '5. Resize semua gambar ke 224×224px',
    '6. Augmentasi kelas dengan gambar < rata-rata',
    '7. Stratified split 70/15/15 (train/val/test)',
]
for s in steps:
    print(f'  {s}')

print('\n✅ EDA selesai! Lanjut ke Step 3: Preprocessing & Penggabungan Dataset')